# Mestres do XGH — Aula 13
## Automação de verdade: do notebook para o script que roda sozinho

**Shift + Enter** roda a célula.

A dívida de sábado passado: *"isso só roda quando eu aperto o play. E se eu quiser que rode sozinho?"*

> **A frase da aula:** script agendado erra sozinho, de madrugada, sem ninguém olhando.

⚠️ **Hoje o notebook é a bancada, não o produto.** A gente vai escrever arquivos `.py` daqui de
dentro e rodá-los como programa. Quem estiver no Colab consegue acompanhar tudo menos a última
parte (agendar) — para isso precisa da máquina local.

---
# 0. Criando os arquivos da aula

Rode esta célula primeiro. Ela cria a pasta `Downloads_teste` com 12 arquivos falsos
e deixa duas ferramentas prontas: `criar_downloads()` e `contar()`.

O `random.seed(13)` garante que os arquivos saem **sempre iguais** — seus números vão bater
com os meus.

In [ ]:
import random, shutil, csv
from pathlib import Path

PASTA = Path("Downloads_teste")

ARQUIVOS = [
    "nota-fiscal-santos.pdf", "contrato-guaruja.pdf", "boleto-novembro.pdf",
    "praia-grande.jpg", "sao-vicente.jpg", "canal-6.png",
    "vendas-outubro.csv", "clientes.csv", "estoque.xlsx",
    "instalador.exe", "anotacoes.txt", "musica-do-verao.mp3",
]


def criar_downloads():
    """Apaga e recria a pasta de teste — sempre com os mesmos 12 arquivos."""
    random.seed(13)
    if PASTA.exists():
        shutil.rmtree(PASTA)
    PASTA.mkdir()
    for nome in ARQUIVOS:
        recheio = "x" * random.randint(50, 400)
        (PASTA / nome).write_text(f"conteudo falso de {nome}\n{recheio}\n", encoding="utf-8")
    print(f"Downloads_teste: {len(ARQUIVOS)} arquivos criados")


def contar():
    """Mede a pasta. É isto que prova se o script é seguro."""
    soltos = [x for x in PASTA.iterdir() if x.is_file()]
    tudo   = [x for x in PASTA.rglob("*") if x.is_file()]
    print(f"soltos na raiz: {len(soltos)}   |   arquivos no total: {len(tudo)}")
    for sub in sorted(x for x in PASTA.iterdir() if x.is_dir()):
        print(f"  {sub.name}/: {len(list(sub.iterdir()))}")


criar_downloads()
contar()

---
# 1. Por que não dá para agendar um notebook

| O que você tem hoje | O que é automação |
|---|---|
| Você abre | Abre sozinho |
| Você aperta o play | Roda no horário |
| Você olha a resposta na tela | Deixa registro de que rodou |
| Se der erro, você vê | Se der erro, **alguém precisa ficar sabendo** |

Notebook **não tem começo nem fim**. Você roda a célula 3, volta na 1, roda a 7, muda a 3 e roda
de novo. O resultado depende da ordem em que você apertou.

Um programa que roda sozinho precisa de **uma** ordem: de cima para baixo, uma vez.

Isso não desmerece o notebook:

> Notebook é **bancada**: você experimenta, olha o dado, erra, corrige.
> O `.py` é o **produto**. Você experimenta no notebook e entrega no `.py`.

---
# 2. Escrevendo o primeiro `.py`

A célula abaixo **grava um arquivo** chamado `organizador_v1.py` na mesma pasta do notebook.
Depois a gente roda ele como programa.

Esta primeira versão é a **ingênua, de propósito**. Leia o código — ele parece certo.

In [ ]:
organizador_v1 = '''
import shutil
from pathlib import Path

PASTA = Path("Downloads_teste")

GRUPOS = {
    "PDFs":      [".pdf"],
    "Imagens":   [".jpg", ".jpeg", ".png"],
    "Planilhas": [".csv", ".xlsx"],
}

def grupo_de(sufixo):
    for nome, extensoes in GRUPOS.items():
        if sufixo.lower() in extensoes:
            return nome
    return "Outros"

movidos = 0
for item in PASTA.iterdir():
    if item.is_dir():
        continue
    destino = PASTA / grupo_de(item.suffix)
    destino.mkdir(exist_ok=True)
    shutil.move(item, destino / item.name)
    movidos += 1

print(f"{movidos} arquivos movidos")
'''

Path("organizador_v1.py").write_text(organizador_v1, encoding="utf-8")
print("organizador_v1.py gravado")

Três coisas para reparar, e as três são a **Aula 8** voltando:

- `Path` e `iterdir()` no lugar de `os.listdir` — mesma ideia, interface melhor
- `item.suffix` já devolve `".pdf"`; não precisa de `split(".")`
- `mkdir(exist_ok=True)` — *"cria se não existir"*. **Guarde essa linha.** Ela já é idempotência
  e a gente volta nela em quinze minutos

### Agora rode como programa

No terminal você digitaria `python organizador_v1.py`. Aqui no notebook, o `!` na frente
faz exatamente isso:

In [ ]:
!python organizador_v1.py

In [ ]:
contar()

Doze entraram, doze saíram — só em pastas diferentes.

**Guarde o número doze.** A gente volta nele.

> Repare no que acabou de acontecer: você **não abriu nada**. Um comando, o programa rodou, fez o
> trabalho e morreu. O agendador sabe fazer exatamente isso: digitar um comando.

---
# 3. O porteiro: `if __name__ == "__main__"`

Primeiro o acidente. Dois arquivos:

In [ ]:
Path("saudacao.py").write_text('''
def saudar(nome):
    return f"bom dia, {nome}"

print("TESTANDO:", saudar("Guilherme"))
''', encoding="utf-8")

Path("usa.py").write_text('''
from saudacao import saudar
print(saudar("turma"))
''', encoding="utf-8")

print("gravados")

In [ ]:
!python usa.py

```
TESTANDO: bom dia, Guilherme
bom dia, turma
```

Eu só queria **usar** a função. O teste que estava no fim do arquivo rodou junto.

Agora imagine que no lugar daquele `print` houvesse um `shutil.move`, ou um `requests.post`.

> **Importar um arquivo executa o arquivo inteiro.**

### A causa, em uma variável

In [ ]:
Path("quem.py").write_text('print("__name__ aqui vale:", __name__)\n', encoding="utf-8")

print("chamado direto:")
!python quem.py
print()
print("importado:")
!python -c "import quem"

```
chamado direto:
__name__ aqui vale: __main__

importado:
__name__ aqui vale: quem
```

O Python coloca a variável `__name__` em todo arquivo:

- chamado **direto** → vale `"__main__"`
- **importado** → vale o nome do arquivo

É a única diferença. E daí sai o porteiro:

```python
if __name__ == "__main__":
    print("TESTANDO:", saudar("Guilherme"))
```

Leia em voz alta, em português: *se eu sou o programa principal, faça isto.*

In [ ]:
Path("saudacao.py").write_text('''
def saudar(nome):
    return f"bom dia, {nome}"

if __name__ == "__main__":
    print("TESTANDO:", saudar("Guilherme"))
''', encoding="utf-8")

print("python saudacao.py  ->")
!python saudacao.py
print()
print("python usa.py       ->")
!python usa.py

### 🔑 A regra para a vida

- em cima do arquivo, as **funções** (`def`)
- embaixo, dentro do `if __name__ == "__main__"`, o que **acontece**

Você vai ver isso em praticamente todo script Python que encontrar. Agora sabe por quê.

---
# 4. Receber a pasta de fora: `sys.argv`

O organizador tem a pasta **escrita dentro do código**. Se o João quiser usar, ele tem que editar
o seu programa. Isso está errado.

`sys.argv` é o que você digitou no terminal, virado em **lista de strings** — a Aula 2 serve aqui:

In [ ]:
Path("argumentos.py").write_text('''
import sys
print("sys.argv =", sys.argv)
''', encoding="utf-8")

!python argumentos.py
!python argumentos.py Downloads_teste --simular

```
sys.argv = ['argumentos.py']
sys.argv = ['argumentos.py', 'Downloads_teste', '--simular']
```

- a posição **0** é sempre o nome do arquivo
- o que você digitou depois vem a partir da posição **1**

Repare: `--simular` entrou na lista como uma string qualquer. **O Python não sabe o que é opção e
o que é pasta — isso é você que decide.** E a gente vai decidir na seção 9.

---
# 5. 💥 A pergunta da aula: *o que acontece se isto rodar duas vezes?*

Este é o bloco mais importante do dia.

O erro que vem agora não dá stack trace, não aparece em log nenhum e não deixa tela vermelha.
O programa **funciona** — e é exatamente por isso que dói.

## 5.1 — O caso fácil de ver: o relatório que duplica

Um script que acrescenta as vendas do dia num `.csv`. Não tem nada de errado nele:

In [ ]:
Path("relatorio.py").write_text('''
import csv
from pathlib import Path

AQUI = Path(__file__).parent
vendas = [
    {"loja": "Centro",   "pecas": 41},
    {"loja": "Shopping", "pecas": 58},
    {"loja": "Online",   "pecas": 33},
]

with open(AQUI / "relatorio.csv", "a", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    for v in vendas:
        w.writerow([v["loja"], v["pecas"]])

print("relatorio.csv atualizado")
''', encoding="utf-8")

Path("relatorio.csv").write_text("loja,pecas\n", encoding="utf-8")
print("relatorio.py gravado, relatorio.csv zerado")

In [ ]:
!python relatorio.py
!python relatorio.py

print()
print(Path("relatorio.csv").read_text(encoding="utf-8"))

```
loja,pecas
Centro,41
Shopping,58
Online,33
Centro,41
Shopping,58
Online,33
```

### Agora **meça** — com o pandas da Aula 10

Número convence; texto não.

In [ ]:
import pandas as pd

df = pd.read_csv("relatorio.csv")
print("linhas:", len(df))
print("total de pecas:", df["pecas"].sum())

```
linhas: 6
total de pecas: 264
```

O total verdadeiro é **132**. O relatório diz **264**.

Nenhum erro, nenhum aviso. Só um número errado, com cara de certo.

> Na **Aula 10** a gente viu dado sujo derrubar um modelo de 100% para 25% **sem dar erro**.
> Aqui é a mesma família de problema: o programa não reclama, o resultado mente.
> A diferença é que esse aqui **você** causou — e você pode consertar.

## 5.2 — O caso difícil de ver (e o mais perigoso)

"Tá, mas o meu organizador não acrescenta nada em arquivo nenhum. Ele só move."

Então vamos rodar duas vezes. Primeiro recriando a pasta:

In [ ]:
criar_downloads()
print()
!python organizador_v1.py
!python organizador_v1.py
print()
contar()

```
12 arquivos movidos
0 arquivos movidos

soltos na raiz: 0   |   arquivos no total: 12
```

Zero movidos, doze arquivos, nada aconteceu. Parece seguro.

> **Não está seguro. Está com sorte.** Ele não duplicou porque na segunda vez não sobrou nada
> solto na raiz para mover. Foi o acaso que salvou, não o código.

### O cenário real

Você baixou **outra** nota fiscal, com o mesmo nome de uma que já foi arquivada. Acontece toda
semana com `nota-fiscal.pdf`, `boleto.pdf`, `extrato.pdf`.

In [ ]:
# chegou um arquivo novo, com um nome que já existe lá dentro
(PASTA / "nota-fiscal-santos.pdf").write_text("NOTA DE NOVEMBRO - outro arquivo, mesmo nome\n", encoding="utf-8")

print("ANTES:")
contar()
print()
!python organizador_v1.py
print()
print("DEPOIS:")
contar()

```
ANTES:
soltos na raiz: 1   |   arquivos no total: 13

1 arquivos movidos

DEPOIS:
soltos na raiz: 0   |   arquivos no total: 12
```

## **Treze arquivos entraram. Doze saíram.**

Um arquivo seu deixou de existir. Não foi para a lixeira, não deu erro, não tem log.
O `shutil.move` escreveu em cima do que já estava lá e ficou quieto.

É o tipo de bug que você descobre três meses depois, quando precisa da nota antiga.

---
# 6. Idempotência

> **IDEMPOTENTE** — rodar uma vez ou dez vezes dá o mesmo resultado.

O nome é feio e você vai ouvir em entrevista de emprego. A ideia é a coisa mais simples do mundo:

- **botão do elevador** → apertar dez vezes é igual a apertar uma. É idempotente
- **interruptor de luz** → apertar duas vezes desliga. Não é

### Antes de agendar qualquer coisa, uma pergunta:

## *o que acontece se isto rodar duas vezes?*

E agendamento **vai** rodar duas vezes. Você testa na mão, o agendador roda no horário, a máquina
hiberna e ele roda ao acordar, você clica em "Executar" para conferir. Duas vezes é o normal,
não a exceção.

### As duas saídas

| Estratégia | Como é na prática |
|---|---|
| **1. Verificar antes de agir** | "já existe? já foi feito? então não faço" |
| **2. Marcar o que já foi feito** | uma lista, uma coluna de data, um nome de arquivo |

O `mkdir(exist_ok=True)` da seção 2 já era a estratégia 1 — você só não sabia o nome.

### Consertando o relatório com a estratégia 1

Pouco código, efeito grande. O `set` é o da Aula 2, o `DictReader` é o da Aula 8:

In [ ]:
Path("relatorio_ok.py").write_text('''
import csv
from pathlib import Path

AQUI = Path(__file__).parent
ARQUIVO = AQUI / "relatorio_ok.csv"
DIA = "2026-11-07"

vendas = [
    {"loja": "Centro",   "pecas": 41},
    {"loja": "Shopping", "pecas": 58},
    {"loja": "Online",   "pecas": 33},
]

# verificar antes de agir: o dia de hoje já está no arquivo?
ja_tem = set()
if ARQUIVO.exists():
    with open(ARQUIVO, newline="", encoding="utf-8") as f:
        for linha in csv.DictReader(f):
            ja_tem.add(linha["dia"])

if DIA in ja_tem:
    print(f"{DIA} ja estava no relatorio - nao fiz nada")
else:
    novo = not ARQUIVO.exists()
    with open(ARQUIVO, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if novo:
            w.writerow(["dia", "loja", "pecas"])
        for v in vendas:
            w.writerow([DIA, v["loja"], v["pecas"]])
    print(f"{DIA} acrescentado - {len(vendas)} linhas")
''', encoding="utf-8")

Path("relatorio_ok.csv").unlink(missing_ok=True)

for i in (1, 2, 3):
    print(f"--- rodada {i} ---")
    !python relatorio_ok.py

In [ ]:
df = pd.read_csv("relatorio_ok.csv")
print("linhas:", len(df), "| total de pecas:", df["pecas"].sum())

```
--- rodada 1 ---
2026-11-07 acrescentado - 3 linhas
--- rodada 2 ---
2026-11-07 ja estava no relatorio - nao fiz nada
--- rodada 3 ---
2026-11-07 ja estava no relatorio - nao fiz nada

linhas: 3 | total de pecas: 132
```

Três rodadas, três linhas, **132** peças. **Agora pode agendar.**

Não tem nada novo de Python nesse conserto. O que é novo é a **pergunta** que te fez escrever ele.

---
# 7. 💥 Funciona no terminal, falha no agendador

A cena é universal: você escreve o script, roda no terminal, funciona. Agenda. Na manhã seguinte
não fez nada — ou fez em algum lugar que você não encontra. Você roda na mão de novo e **funciona**.

A causa tem nome: **diretório de trabalho**.

Todo programa que roda tem uma pasta onde ele "está". Quando você digita `python organizador.py`,
essa pasta é a que você estava no terminal. Quando o **agendador** roda, não é — ele escolhe
outra, e a escolha não é óbvia.

Veja os dois caminhos:

In [ ]:
import os

print("diretorio de trabalho :", os.getcwd())

No notebook o diretório de trabalho é a pasta do próprio notebook. Num script agendado, não.

### 💥 E aqui está a primeira coisa que separa bancada de produto

A solução para script é `Path(__file__).parent` — a pasta onde o **arquivo** mora.
Rode a célula abaixo **dentro do notebook** e veja o que acontece:

In [ ]:
from pathlib import Path

print(Path(__file__).parent)

```
NameError: name '__file__' is not defined. Did you mean: '__name__'?
```

**No notebook não existe `__file__`.** Não tem arquivo `.py` nenhum rodando — tem células.
É mais uma coisa que separa bancada de produto.

Dentro de um `.py`, funciona. É só uma linha, e é a solução inteira:

```python
from pathlib import Path

AQUI = Path(__file__).parent          # onde o SCRIPT mora
ARQUIVO = AQUI / "relatorio.csv"      # não depende de onde você chamou
```

Veja a diferença com o script de verdade, rodado de dois lugares:

In [ ]:
Path("onde_estou.py").write_text('''
import os
from pathlib import Path

print("diretorio de trabalho :", os.getcwd())
print("onde o script mora    :", Path(__file__).parent)
''', encoding="utf-8")

import subprocess, sys, tempfile

script = str(Path("onde_estou.py").resolve())

print("### rodando daqui, da pasta do script")
print(subprocess.run([sys.executable, script], capture_output=True, text=True).stdout)

print("### rodando de OUTRA pasta - e o que o agendador faz")
print(subprocess.run([sys.executable, script], cwd=tempfile.gettempdir(),
                     capture_output=True, text=True).stdout)

O `subprocess.run` aí é só para simular o agendador: ele roda o mesmo script a partir de **outra
pasta**. Na minha máquina o resultado foi:

```
### rodando daqui, da pasta do script
diretorio de trabalho : /home/guilherme/scripts/organizador
onde o script mora    : /home/guilherme/scripts/organizador

### rodando de OUTRA pasta - e o que o agendador faz
diretorio de trabalho : /tmp
onde o script mora    : /home/guilherme/scripts/organizador
```

Os dois eram iguais e agora não são. E `Path("Downloads_teste")` é um caminho **relativo**:
significa *"a partir do diretório de trabalho"*. Se o diretório mudou, o caminho aponta para
outro lugar — e você ganha isto:

```
Traceback (most recent call last):
  File "/home/guilherme/scripts/organizador/organizador_v1.py", line 19, in <module>
    for item in PASTA.iterdir():
  File "/usr/lib/python3.11/pathlib.py", line 931, in iterdir
    for name in os.listdir(self):
                ^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'Downloads_teste'
```

`FileNotFoundError` é da Aula 9, você já leu esse. E repare na última linha: o nome aparece
**sem caminho nenhum**, `'Downloads_teste'`.

> 🔑 **Quando o erro de arquivo não tem caminho na frente, desconfie de caminho relativo na hora.**

### 🔑 A regra, em três linhas

| Que caminho é | O que usar |
|---|---|
| arquivo **do próprio projeto** | `Path(__file__).parent / "..."` |
| caminho que **o usuário passou** | `Path(x).expanduser().resolve()` |
| caminho relativo em script agendado | **nunca** |

O `expanduser()` troca o `~` pela pasta do usuário; o `resolve()` deixa o caminho absoluto.

---
# 8. 💥 Morrer calado

O script roda às 3h, dá pau no terceiro arquivo e morre. Quem ficou sabendo? **Ninguém.**

Seu `print` foi para um terminal que não existe.

| `print` | `logging` |
|---|---|
| vai para a tela | vai para **arquivo** |
| sem hora | com **data e hora** |
| tudo igual | tem **nível**: INFO, WARNING, ERROR |
| você apaga depois | **fica** |

A configuração é chamada **uma vez**, no começo do arquivo. Depois é só `logging.info(...)`
em qualquer lugar — não precisa passar nada de função em função.

In [ ]:
import logging

print("""
logging.basicConfig(
    filename=AQUI / "organizador.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%d/%m/%Y %H:%M:%S",
    encoding="utf-8",
)
""")

E a linha que sai **dentro do arquivo** `organizador.log`:

```
22/10/2026 19:04:25 | INFO | movido: anotacoes.txt -> Outros/anotacoes.txt
```

Data, hora, gravidade, mensagem. Quatro campos — é tudo que você precisa às oito da manhã de uma
segunda, quando alguém diz que faltou arquivo.

### O `try/except` da Aula 9 deixa de ser zelo

> Na Aula 9 o `try/except` evitava a tela vermelha. Em script agendado ele é **requisito**, e por
> um motivo novo: sem `except`, o programa para no primeiro arquivo problemático e os outros onze
> não são tratados.

```python
try:
    shutil.move(item, destino)
    logging.info(f"movido: {item.name} -> {sub.name}/{destino.name}")
except Exception as e:
    logging.error(f"falhou em {item.name}: {type(e).__name__}: {e}")
```

Um arquivo deu problema? Anota e **segue para o próximo**. É a diferença entre *"falhou"* e
*"falhou um de doze"*.

### 🔑 O melhor comando da aula: `logging.exception`

Uma palavra de diferença, e o **stack trace inteiro** vai para dentro do arquivo:

```python
except Exception as e:
    logging.exception(f"morreu antes de terminar: {type(e).__name__}: {e}")
```

```
22/10/2026 19:04:28 | INFO | --- inicio  em /home/guilherme/scripts/organizador/Downloads_do_fulano
22/10/2026 19:04:28 | ERROR | morreu antes de terminar: FileNotFoundError: [Errno 2] No such file or directory: '/home/guilherme/scripts/organizador/Downloads_do_fulano'
Traceback (most recent call last):
  File "/home/guilherme/scripts/organizador/organizador.py", line 90, in <module>
    movidos, iguais, erros = organizar(pasta, simulacao)
                             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/guilherme/scripts/organizador/organizador.py", line 45, in organizar
    for item in sorted(pasta.iterdir()):
                ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/pathlib.py", line 931, in iterdir
    for name in os.listdir(self):
                ^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/home/guilherme/scripts/organizador/Downloads_do_fulano'
```

Aquele mesmo stack trace que você aprendeu a ler na Aula 9, agora esperando por você de manhã.

> **Regra:** `logging.exception(...)` dentro do `except` de nível mais alto do seu script. Sempre.

---
# 9. 🛠️ Mão na massa: o organizador de verdade

Você vai sair daqui com um script que:

1. recebe a pasta de `sys.argv` — e se não vier, explica como se usa e sai
2. usa `AQUI = Path(__file__).parent` para o log
3. escreve `logging` em arquivo, com `try/except` por arquivo e `logging.exception` no fim
4. é **idempotente**

### Antes de codar, decida a nº 4 por escrito

Para ser idempotente eu preciso responder: **o arquivo já está lá?** E são **três** respostas,
não duas:

| Situação | O que fazer |
|---|---|
| no destino **não existe** nada com esse nome | move, normal |
| existe e é do **mesmo tamanho** | é o mesmo arquivo, já foi feito → **não mexe**, loga |
| existe e é de **outro tamanho** | é outro arquivo com o mesmo nome → move como `nota-1.pdf` |

O segundo e o terceiro caso são os dois que o `shutil.move` resolvia do mesmo jeito: **apagando**.
Agora um não faz nada e o outro guarda os dois. **Nenhum arquivo seu morre.**

> Tamanho é uma checagem fraca — dois arquivos diferentes podem ter o mesmo tamanho. O jeito certo
> é comparar o conteúdo com `hashlib`. Hoje tamanho basta; no Projeto 2, troque.

### A única parte nova de Python do dia

```python
def nome_livre(destino):
    """Acha o próximo nome livre: nota.pdf -> nota-1.pdf -> nota-2.pdf ..."""
    if not destino.exists():
        return destino
    n = 1
    while True:
        tentativa = destino.with_name(f"{destino.stem}-{n}{destino.suffix}")
        if not tentativa.exists():
            return tentativa
        n += 1
```

`while True` com `return` dentro é o `while` da Aula 4. `stem` é o nome sem extensão,
`suffix` é a extensão. É só isso.

## 9.1 — O script completo

Grave e leia com calma. **Depois digite o seu**, não copie: na Aula 14 você vai ter que mexer nele.

In [ ]:
organizador = '''
import logging
import shutil
import sys
from pathlib import Path

AQUI = Path(__file__).parent          # onde o SCRIPT mora - nao depende do terminal

logging.basicConfig(
    filename=AQUI / "organizador.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%d/%m/%Y %H:%M:%S",
    encoding="utf-8",
)

GRUPOS = {
    "PDFs":      [".pdf"],
    "Imagens":   [".jpg", ".jpeg", ".png"],
    "Planilhas": [".csv", ".xlsx"],
}


def grupo_de(sufixo):
    for nome, extensoes in GRUPOS.items():
        if sufixo.lower() in extensoes:
            return nome
    return "Outros"


def nome_livre(destino):
    """Acha o proximo nome livre: nota.pdf -> nota-1.pdf -> nota-2.pdf ..."""
    if not destino.exists():
        return destino
    n = 1
    while True:
        tentativa = destino.with_name(f"{destino.stem}-{n}{destino.suffix}")
        if not tentativa.exists():
            return tentativa
        n += 1


def organizar(pasta, simulacao=False):
    movidos = iguais = erros = 0

    for item in sorted(pasta.iterdir()):
        if item.is_dir() or item.suffix == ".log":
            continue

        sub = pasta / grupo_de(item.suffix)
        destino = sub / item.name

        # 1) ja existe um arquivo igual la? entao isto ja foi feito.
        if destino.exists() and destino.stat().st_size == item.stat().st_size:
            logging.info(f"ja feito, nao mexi: {item.name}")
            iguais += 1
            continue

        # 2) existe com o mesmo nome mas e outro arquivo? guarda os dois.
        destino = nome_livre(destino)

        if simulacao:
            print(f"[SIMULACAO] moveria  {item.name}  ->  {sub.name}/{destino.name}")
            continue

        try:
            sub.mkdir(exist_ok=True)
            shutil.move(item, destino)
            logging.info(f"movido: {item.name} -> {sub.name}/{destino.name}")
            movidos += 1
        except Exception as e:
            logging.error(f"falhou em {item.name}: {type(e).__name__}: {e}")
            erros += 1

    return movidos, iguais, erros


if __name__ == "__main__":
    argumentos = sys.argv[1:]
    simulacao = "--simular" in argumentos
    caminhos = [a for a in argumentos if not a.startswith("--")]

    if not caminhos:
        print("uso: python organizador.py <pasta> [--simular]")
        sys.exit(1)

    pasta = Path(caminhos[0]).expanduser().resolve()
    modo = "(SIMULACAO)" if simulacao else ""
    logging.info(f"--- inicio {modo} em {pasta}")

    try:
        movidos, iguais, erros = organizar(pasta, simulacao)
        print(f"movidos: {movidos} | ja estavam feitos: {iguais} | erros: {erros}")
        logging.info(f"--- fim: {movidos} movidos, {iguais} iguais, {erros} erros")
    except Exception as e:
        logging.exception(f"morreu antes de terminar: {type(e).__name__}: {e}")
        print("deu erro - olhe o organizador.log")
        sys.exit(1)
'''

Path("organizador.py").write_text(organizador, encoding="utf-8")
print("organizador.py gravado")

## 9.2 — ⭐ O modo simulação

A parte que eu mais quero que você leve desta aula, e não é sintaxe. É um **hábito**.

```python
if simulacao:
    print(f"[SIMULACAO] moveria  {item.name}  ->  {sub.name}/{destino.name}")
    continue
```

Rode e repare: doze linhas de *"eu faria assim"*, e **nada muda na pasta**.

In [ ]:
criar_downloads()
print()
!python organizador.py Downloads_teste --simular
print()
contar()

```
[SIMULACAO] moveria  anotacoes.txt  ->  Outros/anotacoes.txt
[SIMULACAO] moveria  boleto-novembro.pdf  ->  PDFs/boleto-novembro.pdf
[SIMULACAO] moveria  canal-6.png  ->  Imagens/canal-6.png
[SIMULACAO] moveria  clientes.csv  ->  Planilhas/clientes.csv
[SIMULACAO] moveria  contrato-guaruja.pdf  ->  PDFs/contrato-guaruja.pdf
[SIMULACAO] moveria  estoque.xlsx  ->  Planilhas/estoque.xlsx
[SIMULACAO] moveria  instalador.exe  ->  Outros/instalador.exe
[SIMULACAO] moveria  musica-do-verao.mp3  ->  Outros/musica-do-verao.mp3
[SIMULACAO] moveria  nota-fiscal-santos.pdf  ->  PDFs/nota-fiscal-santos.pdf
[SIMULACAO] moveria  praia-grande.jpg  ->  Imagens/praia-grande.jpg
[SIMULACAO] moveria  sao-vicente.jpg  ->  Imagens/sao-vicente.jpg
[SIMULACAO] moveria  vendas-outubro.csv  ->  Planilhas/vendas-outubro.csv
movidos: 0 | ja estavam feitos: 0 | erros: 0

soltos na raiz: 12   |   arquivos no total: 12
```

Esse modo tem nome em inglês — **dry run** — e é a coisa mais profissional desta aula.
Em ferramenta séria ele sempre existe: o `git` tem `--dry-run`, o `rsync` tem `-n`,
instalador de pacote tem.

E o motivo é bem pouco nobre:

> **Ninguém confia em programa que apaga coisa.** Nem você, no seu próprio.
> O modo simulação é o que te dá coragem de apontar o script para a sua pasta de Downloads de
> verdade em vez da de teste.

### 🔑 Regra

**Script que mexe em arquivo nasce com `--simular`. E o `--simular` é a primeira coisa que você
roda quando aponta para uma pasta nova.**

## 9.3 — A prova: rodando de verdade, três vezes

Agora sem simulação. Repare nos números de cada rodada — e no total de arquivos, que **nunca cai**.

In [ ]:
print("### rodada 1")
!python organizador.py Downloads_teste
contar()
print()
print("### rodada 2 - sem mudar nada")
!python organizador.py Downloads_teste
contar()

```
### rodada 1
movidos: 12 | ja estavam feitos: 0 | erros: 0
soltos na raiz: 0   |   arquivos no total: 12

### rodada 2 - sem mudar nada
movidos: 0 | ja estavam feitos: 0 | erros: 0
soltos na raiz: 0   |   arquivos no total: 12
```

### E agora o teste que o `organizador_v1` reprovou

Chega outra nota fiscal com o mesmo nome. Lembre: na versão ingênua, 13 entraram e 12 saíram.

In [ ]:
(PASTA / "nota-fiscal-santos.pdf").write_text("NOTA DE NOVEMBRO - outro arquivo, mesmo nome\n", encoding="utf-8")
print("ANTES:")
contar()
print()
!python organizador.py Downloads_teste
print()
print("DEPOIS:")
contar()
print()
print("dentro de PDFs/:", sorted(p.name for p in (PASTA / "PDFs").iterdir()))

```
ANTES:
soltos na raiz: 1   |   arquivos no total: 13

movidos: 1 | ja estavam feitos: 0 | erros: 0

DEPOIS:
soltos na raiz: 0   |   arquivos no total: 13

dentro de PDFs/: ['boleto-novembro.pdf', 'contrato-guaruja.pdf', 'nota-fiscal-santos-1.pdf', 'nota-fiscal-santos.pdf']
```

**Treze entraram, treze saíram.** Os dois arquivos estão lá, com nomes diferentes.

### E o log, que é o que vai te salvar de madrugada

In [ ]:
print(Path("organizador.log").read_text(encoding="utf-8"))

Uma linha por arquivo, com hora. Mais as linhas de `--- inicio` e `--- fim` de cada rodada,
que são o que te deixa responder *"rodou? a que horas? fez o quê?"* sem adivinhar.

---
# 10. Agendar: o Agendador de Tarefas do Windows

### Primeiro, descubra o caminho do seu Python

O agendador **não tem o seu PATH**. Você não pode escrever só `python` lá — tem que ser o caminho
inteiro do `python.exe`. Isto é a Aula 11 voltando:

In [ ]:
import sys
print(sys.executable)

No Windows isso sai parecido com:

```
C:\Users\guilherme\AppData\Local\Programs\Python\Python312\python.exe
```

No terminal, `where python` também mostra. **Copie para um bloco de notas agora.**

---

### O passo a passo

**1.** `Win + R` → digite `taskschd.msc` → Enter

**2.** No painel da direita, **"Criar Tarefa..."** — *não* use "Criar Tarefa Básica".
A básica não tem o campo que importa.

**3.** Aba **Geral** → Nome: `Organizador de Downloads`.
Deixe marcado *"Executar somente quando o usuário estiver conectado"*.

**4.** Aba **Disparadores** → **Novo...** → `Diariamente`, hora `22:00` → OK

**5.** Aba **Ações** → **Novo...** → Ação: `Iniciar um programa`. Os **três campos que decidem tudo**:

| Campo | O que vai |
|---|---|
| **Programa/script** | `C:\Users\guilherme\AppData\Local\Programs\Python\Python312\python.exe` |
| **Adicionar argumentos** | `organizador.py "C:\Users\guilherme\Downloads"` |
| **Iniciar em (opcional)** | `C:\Users\guilherme\scripts\organizador` |

**6.** OK, OK. A tarefa aparece na lista.

**7.** Clique com o **botão direito nela → "Executar"**. Não espere as 22h para descobrir que
está errado.

**8.** Abra o `organizador.log` na pasta do script. Linha nova? Funcionou.

---

### ⚠️ O campo 3 diz "opcional" e não é

> **"Iniciar em (opcional)"** é o **diretório de trabalho**. Se você deixar vazio, o Windows escolhe
> `C:\Windows\System32`, e todo caminho relativo do seu script aponta para lá.

É literalmente o erro da seção 7, com a palavra "opcional" do lado para te enganar.

E repare nas **aspas** em volta do caminho nos argumentos: é por causa do espaço em
"Program Files", em "Meus Documentos", no seu nome. **Caminho do Windows tem espaço — sempre
ponha aspas.**

---

### Como ler o resultado

Selecione a tarefa na lista e olhe as colunas:

- **Última Execução** — a hora
- **Resultado da Última Execução** — `0x0` quer dizer que o programa terminou sem erro.
  Qualquer outra coisa quer dizer que terminou com erro
- a aba **Histórico** pode estar desligada; tem um botão *"Habilitar histórico de todas as tarefas"*

> `0x0` é o sucesso. Mas cuidado com o que ele significa: quer dizer que o **programa terminou**,
> não que fez a coisa certa. É o `status 200` da Aula 12 de novo — sucesso na entrega não é sucesso
> no conteúdo. **Quem conta a verdade é o seu log.**

---

### A janelinha preta piscando

Troque `python.exe` por `pythonw.exe`, na mesma pasta. O `w` é de *windowless*.
Mas só **depois** que o log estiver funcionando — sem janela e sem log você fica cego.

### O caminho mais robusto: um `.bat`

```bat
@echo off
cd /d "C:\Users\guilherme\scripts\organizador"
"C:\Users\guilherme\AppData\Local\Programs\Python\Python312\python.exe" organizador.py "C:\Users\guilherme\Downloads"
```

Aponte o agendador para o `.bat` e pronto. A vantagem: o `cd /d` resolve o diretório de trabalho
num arquivo que está **no seu projeto** e que você **vê** — não num campo de uma janela do Windows
que ninguém lembra de conferir.

### O equivalente em Linux e Mac

Metade dos tutoriais da internet está nessa língua, então vale reconhecer:

```
crontab -e
```

```
0 22 * * * /usr/bin/python3 /home/guilherme/scripts/organizador.py /home/guilherme/Downloads
```

`0 22 * * *` é *"minuto 0 da hora 22, todo dia"*. Mesma ideia, mesma armadilha: o `cron` tem um
ambiente ainda mais pelado que o do Windows — por isso ali também vai o caminho completo do
`python3`.

---

### ⚠️ Antes de agendar na pasta de verdade

Você acabou de ganhar a capacidade de fazer o computador mexer nos seus arquivos sozinho,
de madrugada.

1. agende primeiro apontando para uma **pasta de teste**
2. rode com `--simular`
3. olhe o log
4. **só depois** aponte para a pasta de verdade

---
# 11. O que isso destrava

Com `.py` + argumento + log + agendador, estas coisas saíram do *"um dia eu faço"*:

- **relatório que chega pronto** toda segunda, sem você abrir nada
- **backup** de uma pasta, no horário, com registro do que foi copiado
- a **cotação da Aula 12** anotada num `.csv` de hora em hora, virando histórico seu
- **monitor de preço** que avisa quando cai
- e, do Módulo 4 em diante, o mesmo esqueleto chamando **IA** em cima dos seus arquivos

> Até ontem o computador fazia o que você **pedia**.
> De hoje em diante ele faz o que você **combinou**.

É outro patamar — e é por isso que a aula teve três erros de propósito em vez de um.

---
# 12. 📝 Tarefa da semana

**1.** **Terminar o organizador** com as quatro exigências: `sys.argv`,
`Path(__file__).parent`, `logging` em arquivo e `--simular`.

**2.** **Rodar duas vezes seguidas e provar com número** que nada duplicou e nada desapareceu.
Conte os arquivos antes e depois e **mande os dois números no grupo**.

**3.** **Agendar de verdade** no Agendador de Tarefas, apontando para uma **pasta de teste**,
e mandar no grupo o `organizador.log` com as linhas das duas execuções.

**4.** **Pensar** (não codar) na sua automação pessoal: o que você faz na mão toda semana e cabe
num script? Traga a ideia **escrita** — ela vira o Projeto 2 sábado que vem.

**5. Desafio:** fazer o script **apagar** a cópia duplicada quando o arquivo é idêntico ao que já
está arquivado. Só vale se funcionar com `--simular` **primeiro**.

---

## A pergunta que fica

Você vai passar a semana mexendo nesse script. Vai consertar uma coisa, quebrar outra, voltar
atrás. E em algum momento — eu garanto — ele vai parar de funcionar e **você não vai saber o que
mudou**.

Aí vem o `organizador_v2.py`, o `organizador_FINAL.py`, o `organizador_FINAL_agora_vai.py`.
Todo mundo já fez isso.

**Sábado que vem a gente resolve isso de verdade — e de graça.**

---

*Mestres do XGH · Aula 13 · 07/11/2026*

In [ ]:
# Seu espaço — desafio 5: apagar a cópia idêntica, com --simular primeiro
